# Social media engagement prediction (pre-publish)

Predict whether a planned post will be “high engagement” (top quartile of `engagement_rate` within its platform), using only pre-publish fields.

- **Label**: `is_high_engagement` (binary)
- **Leakage avoidance**: time-based split by `created_at`
- **Output**: `ml-outputs/social-engagement-prediction/predictions.csv` (holdout predictions + recommended windows)

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')

DATA_DIR = '../lighthouse_csv_v7'
OUT_DIR = '../ml-outputs/social-engagement-prediction'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

label_source_col = 'engagement_rate'
date_col = 'created_at'

prepublish_categorical = [
    'platform', 'day_of_week', 'post_type', 'media_type',
    'has_call_to_action', 'call_to_action_type',
    'content_topic', 'sentiment_tone',
    'features_resident_story',
    'campaign_name',
    'is_boosted',
]

prepublish_numeric = [
    'post_hour', 'caption_length', 'num_hashtags', 'mentions_count',
    'boost_budget_php', 'follower_count_at_post',
]

print('Output folder:', OUT_DIR)

In [ ]:
social = pd.read_csv(os.path.join(DATA_DIR, 'social_media_posts.csv'))
social[date_col] = pd.to_datetime(social[date_col], errors='coerce', utc=True)

if label_source_col not in social.columns:
    raise ValueError(f"Expected '{label_source_col}' in social_media_posts.csv")

# Keep valid rows
df = social.dropna(subset=[label_source_col, date_col, 'platform']).copy()
df[label_source_col] = pd.to_numeric(df[label_source_col], errors='coerce')
df = df.dropna(subset=[label_source_col]).copy()

# Define label as top quartile within platform
p75 = df.groupby('platform')[label_source_col].quantile(0.75).rename('p75')
df = df.join(p75, on='platform')
df['is_high_engagement'] = (df[label_source_col] >= df['p75']).astype(int)

feature_cols = [c for c in (prepublish_categorical + prepublish_numeric) if c in df.columns]

print('Rows:', len(df))
print('Positive rate:', float(df['is_high_engagement'].mean()))
print('Feature cols:', feature_cols)

# Time split
cutoff = df[date_col].quantile(1 - TEST_SIZE)
train_df = df[df[date_col] <= cutoff].copy()
test_df = df[df[date_col] > cutoff].copy()

X_train = train_df[feature_cols]
y_train = train_df['is_high_engagement']
X_test = test_df[feature_cols]
y_test = test_df['is_high_engagement']

print('Train rows:', len(train_df), 'Test rows:', len(test_df))

In [ ]:
numeric_features = [c for c in prepublish_numeric if c in feature_cols]
categorical_features = [c for c in prepublish_categorical if c in feature_cols]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop'
)

models = {
    'LogReg': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(
        n_estimators=400,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight='balanced_subsample',
        min_samples_leaf=2,
    ),
}

rows = []
fitters = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    ap = average_precision_score(y_test, proba)
    rows.append({'model': name, 'roc_auc': auc, 'pr_auc': ap})
    fitters[name] = pipe

results = pd.DataFrame(rows).sort_values(['pr_auc', 'roc_auc'], ascending=False).reset_index(drop=True)
results

In [ ]:
best_name = results.loc[0, 'model']
best = fitters[best_name]

proba = best.predict_proba(X_test)[:, 1]

# Choose a default threshold aiming for reasonable recall
# (simple heuristic: threshold where precision ~ recall, if possible)
prec, rec, ths = precision_recall_curve(y_test, proba)
ths = np.concatenate(([0.0], ths))

gap = np.abs(prec - rec)
best_ix = int(np.nanargmin(gap))
th = float(ths[best_ix])

pred = (proba >= th).astype(int)

print('Best model:', best_name)
print('Default threshold:', th)
print('ROC AUC:', roc_auc_score(y_test, proba))
print('PR AUC:', average_precision_score(y_test, proba))
print('\nClassification report:')
print(classification_report(y_test, pred, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))

out = test_df[['post_id', 'post_url', date_col, 'platform', 'day_of_week', 'post_hour']].copy()
out['y_true'] = y_test.values
out['p_high_engagement'] = proba
out['y_pred'] = pred

out_path = os.path.join(OUT_DIR, 'predictions.csv')
out.to_csv(out_path, index=False)
print('Wrote:', out_path)

# Recommended windows: average predicted probability by platform/day/hour
rec_table = (out.groupby(['platform', 'day_of_week', 'post_hour'])
              .p_high_engagement.mean()
              .sort_values(ascending=False)
              .head(25)
              .reset_index())
rec_path = os.path.join(OUT_DIR, 'recommended_windows.csv')
rec_table.to_csv(rec_path, index=False)
print('Wrote:', rec_path)

rec_table.head(10)

## Website surfacing + exported CSV schema

### Admin dashboard
- For a draft post: show `p_high_engagement` and the platform-relative definition (“top quartile engagement rate”).
- Use `recommended_windows.csv` to suggest when to post.

### Public impact page (public-safe)
- Show aggregated “best posting windows” (no promises; frame as historical optimization).

### Exported files
- `ml-outputs/social-engagement-prediction/predictions.csv`
  - `post_id`, `post_url`, `created_at`, `platform`, `day_of_week`, `post_hour`, `y_true`, `p_high_engagement`, `y_pred`
- `ml-outputs/social-engagement-prediction/recommended_windows.csv`
  - `platform`, `day_of_week`, `post_hour`, `p_high_engagement` (mean predicted probability in holdout)